# Uber Fare Prediction: Model Comparison
In this notebook, we compare the baseline Linear Regression, HistGradientBoostingRegressor (tuned), Random Forest Regressor, and Decision Tree Regressor.

## Section 0: Imports & Data Loading

In [1]:
import pandas as pd
import numpy as np
import joblib
import pickle
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42

# Load data
df = pd.read_csv('uber_clean.csv')

## Section 1: Data Preparation

In [2]:
# Feature engineering
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

# Drop columns
FEATURES_TO_DROP = ['pickup_datetime', 'jfk_dist', 'ewr_dist', 'lga_dist', 'sol_dist', 'hour']
df_model = df.drop(columns=FEATURES_TO_DROP)

# Winsorize coordinates
coord_cols = ['pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude']
for c in coord_cols:
    lower, upper = df_model[c].quantile(0.005), df_model[c].quantile(0.995)
    df_model[c] = df_model[c].clip(lower, upper)

# Split
X = df_model.drop(columns=['fare_amount'])
y = df_model['fare_amount']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

## Section 2: Define Preprocessor

In [3]:
CAR_CONDITION_ORDER = ['Bad', 'Good', 'Very Good', 'Excellent']
TRAFFIC_ORDER = ['Flow Traffic', 'Dense Traffic', 'Congested Traffic']

ordinal_features = ['Car Condition', 'Traffic Condition']
nominal_features = ['Weather']
numeric_features = ['distance', 'bearing', 'passenger_count', 'day', 'month', 'weekday', 'year',
                    'hour_sin', 'hour_cos', 'nyc_dist',
                    'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])
ordinal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(categories=[CAR_CONDITION_ORDER, TRAFFIC_ORDER])),
])
nominal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore')),
])
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('ord', ordinal_transformer, ordinal_features),
    ('nom', nominal_transformer, nominal_features),
])

## Section 3: Train All Four Models

In [4]:
models = {
    'Linear Regression': Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', LinearRegression())
    ]),
    'Decision Tree Regressor': Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', DecisionTreeRegressor(random_state=RANDOM_STATE))
    ]),
    'Random Forest Regressor': Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE))
    ]),
    'HistGradientBoosting Regressor': Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('feature_selection', SelectFromModel(Lasso(alpha=0.01, random_state=RANDOM_STATE))),
        ('regressor', HistGradientBoostingRegressor(learning_rate=0.1, max_iter=300, max_depth=7, max_leaf_nodes=63, l2_regularization=0.0, random_state=RANDOM_STATE)),
    ])
}

results = []

for name, pipeline in models.items():
    print(f"Training {name}...")
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    results.append({
        'Model': name,
        'MAE': mae,
        'RMSE': rmse,
        'R²': r2
    })

Training Linear Regression...
Training Decision Tree Regressor...
Training Random Forest Regressor...
Training HistGradientBoosting Regressor...


## Section 4: Comparison Table

In [5]:
df_results = pd.DataFrame(results)
df_results.sort_values('RMSE', inplace=True)
display(df_results)

,Model,MAE,RMSE,R²
3,HistGradientBoosting Regressor,1.141790,1.751455,0.821079
2,Random Forest Regressor,1.230923,1.828569,0.804977
0,Linear Regression,1.573945,2.316905,0.686902
1,Decision Tree Regressor,1.764151,2.660125,0.587268


## Section 5: Model Justification
The **HistGradientBoosting Regressor** is chosen as the final model because it delivers the best overall performance among the evaluated options. It achieves the lowest Mean Absolute Error (MAE) and Root Mean Squared Error (RMSE), along with the highest R² score on the test set. This model effectively handles complex, non-linear relationships and interactions within the data. Furthermore, the inclusion of a Lasso feature selection step helps eliminate irrelevant features, leading to a more robust model. Finally, the histogram-based approach offers a highly favorable balance of predictive accuracy and training speed.

## Section 6: Save Best Model

In [6]:
# The best model based on our evaluation is HistGradientBoosting Regressor
best_pipeline = models['HistGradientBoosting Regressor']

# Save as .pkl as required by the task PDF
with open('uber_fare_model.pkl', 'wb') as f:
    pickle.dump(best_pipeline, f)

# Also keep the joblib version
joblib.dump(best_pipeline, 'uber_fare_pipeline.joblib')

print("Model saved successfully!")

Model saved successfully!
